# Rendering showcase

Select the Python kernel from the project's uv environment and run from the top. Each comparison labels its settings and keeps the board fixed where useful. The panels preserve the original pixels. The browser may scale wide rows to fit the notebook.

In [ ]:
from dataclasses import replace
from pathlib import Path
from tempfile import TemporaryDirectory

from IPython.display import Markdown, display
from PIL import Image, ImageDraw, ImageFont, ImageOps

from synthetic_scenes.augmentation import augment_scene
from synthetic_scenes.desktop import wrap_window
from synthetic_scenes.minesweeper.board import generate_game
from synthetic_scenes.minesweeper.drawing import font_for
from synthetic_scenes.minesweeper.renderer import render_board, render_game_region
from synthetic_scenes.sampling import sample_scene
from synthetic_scenes.scene import GameSpec, render_scene


STYLE_NAMES = {
    "classic": "Classic",
    "windows7": "Windows 7",
    "microsoft": "Microsoft",
    "google": "Google",
}


def show_row(*examples):
    """Display labeled Pillow images side by side at native pixel size."""
    examples = [(label.replace("×", "x").replace("→", "->"), image) for label, image in examples]
    font = ImageFont.load_default(size=15)
    gap = 12
    widths = [max(image.width, round(font.getlength(label)) + 16) for label, image in examples]
    width = sum(widths) + gap * (len(examples) - 1)
    height = 30 + max(image.height for _, image in examples)
    preview = Image.new("RGB", (width, height), "#212833")
    draw = ImageDraw.Draw(preview)
    left = 0
    for (label, image), panel_width in zip(examples, widths):
        draw.text((left + 8, 7), label, fill="white", font=font)
        preview.paste(image, (left, 30))
        left += panel_width + gap
    display(preview)


shared_game = generate_game(status="active", progress=.48, seed=42, elapsed_seconds=87)
# Includes flags, untriggered mines, an exploded mine, and a wrong flag.
lost_game = generate_game(status="lost", progress=.85, seed=42, elapsed_seconds=148)

## Grid styles and every theme

The same 9 × 9 position and 24 px tile size are used throughout. Windows 7's `default` and `blue` are aliases, so their panels should look identical.

In [ ]:
theme_groups = {
    "classic": ("default", "blue", "green", "warm"),
    "windows7": ("default", "blue", "green"),
    "microsoft": ("default", "blue", "warm"),
    "google": ("default", "blue", "warm"),
}

for style, themes in theme_groups.items():
    display(Markdown(f"**{STYLE_NAMES[style]}**"))
    examples = []
    for theme in themes:
        image, _ = render_board(shared_game, style=style, theme=theme, cell_size=24)
        examples.append((theme, image))
    if len(examples) == 4:
        show_row(*examples[:2])
        show_row(*examples[2:])
    else:
        show_row(*examples)

## Typography, digit colors, and rare digits

Family, scale, and weight change in separate comparisons. Named families depend on local fonts. The family comparison reports the face Skia resolves.

Each rare digit is rendered on its own targeted board. A pink rectangle locates the target without covering the digit.

In [ ]:
font_families = ("Arial", "Tahoma", "Verdana", "Courier New")
family_examples = []
for family in font_families:
    image, _ = render_board(shared_game, font_family=family, cell_size=25)
    resolved_family = font_for(family, 18).getTypeface().getFamilyName()
    label = family if resolved_family == family else f"{family} → {resolved_family}"
    family_examples.append((label, image))
show_row(*family_examples[:2])
show_row(*family_examples[2:])

scale_examples = []
for scale in (.52, .72, .88):
    image, _ = render_board(shared_game, font_scale=scale, cell_size=25)
    scale_examples.append((f"Arial bold · scale {scale:.2f}", image))
show_row(*scale_examples)

regular, _ = render_board(shared_game, bold=False, cell_size=25)
bold, _ = render_board(shared_game, bold=True, cell_size=25)
show_row(("Arial regular · scale 0.72", regular), ("Arial bold · scale 0.72", bold))

In [ ]:
def highlight_target(image, metadata, position):
    preview = image.copy()
    row, column = position
    left, top, right, bottom = metadata["cell_boxes"][row][column]
    ImageDraw.Draw(preview).rectangle(
        (left, top, right - 1, bottom - 1), outline="#ed287d", width=2,
    )
    return preview


rare_examples = []
for digit in (5, 6, 7, 8):
    game = generate_game(
        status="active", progress=.35, seed=42, mode="targeted", target_number=digit,
    )
    image, metadata = render_board(game, style="classic", cell_size=27)
    rare_examples.append((f"Classic · target {digit}", highlight_target(image, metadata, game["target_position"])))
show_row(*rare_examples[:2])
show_row(*rare_examples[2:])

custom_palette = (
    "#005a6b", "#176a3d", "#a23420", "#543a86",
    "#8a3c58", "#146970", "#30245b", "#8d234b",
)
custom_image, metadata = render_board(game, style="classic", cell_size=27, digit_palette=custom_palette)
show_row(rare_examples[-1], ("Same target 8 · custom palette", highlight_target(custom_image, metadata, game["target_position"])))

## Board scale, tile spacing, and reveal boundaries

Compare the same position at two native Classic tile sizes, then compare a compact board with a wide board at the same tile size. Microsoft spacing follows tile size automatically. Its close-ups show the same upper-left 3 × 3 cells at three native sizes. The Google row isolates the covered-side boundary shading.

In [ ]:
small, _ = render_board(shared_game, cell_size=16)
large, _ = render_board(shared_game, cell_size=42)
show_row(("Classic · 9 × 9 · 16 px", small), ("Classic · 9 × 9 · 42 px", large))

dimension_examples = []
for rows, columns, mines in ((6, 10, 10), (9, 18, 25)):
    game = generate_game(rows=rows, columns=columns, mine_count=mines,
                         status="active", progress=.45, seed=42)
    image, _ = render_board(game, style="classic", cell_size=20)
    dimension_examples.append((f"Classic · {rows} × {columns} · 20 px", image))
show_row(*dimension_examples)

spacing_examples = []
for tile_size in (14, 28, 46):
    image, metadata = render_board(shared_game, style="microsoft", cell_size=tile_size)
    gap = metadata["style_parameters"]["gap"]
    _, _, right, bottom = metadata["cell_boxes"][2][2]
    spacing_examples.append((f"{tile_size} px tiles · {gap} px gaps", image.crop((0, 0, right, bottom))))
show_row(*spacing_examples)

boundary_examples = []
for ratio in (0, .08, .20):
    image, _ = render_board(shared_game, style="google", cell_size=24, boundary_ratio=ratio)
    label = "Off" if ratio == 0 else f"{ratio:.0%} of tile size"
    boundary_examples.append((f"Google boundary · {label}", image))
show_row(*boundary_examples)

## Game states, symbols, and controls

Classic panels show new, active, won, and lost states on the same seeded minefield. Compare their seven-segment counters, timers, and reset faces. The four-style loss comparison uses one position to expose the differences in flags, mines, exploded mines, and crossed-out wrong flags.

Windows 7 also combines both themes with metallic mines and Flower Garden symbols. Compare the board and the mine-counter icon together.

In [ ]:
status_examples = []
for status, elapsed in (("new", 0), ("active", 87), ("won", 203), ("lost", 148)):
    progress = 0 if status == "new" else .85
    game = generate_game(status=status, progress=progress, seed=42, elapsed_seconds=elapsed)
    image, _ = render_game_region(game, style="classic", cell_size=24)
    status_examples.append((f"Classic · {status} · {elapsed} s", image))
show_row(*status_examples[:2])
show_row(*status_examples[2:])

loss_examples = []
for style in ("windows7", "microsoft", "google"):
    image, _ = render_game_region(lost_game, style=style, cell_size=24)
    loss_examples.append((f"{STYLE_NAMES[style]} · lost", image))
show_row(*loss_examples)

for theme in ("blue", "green"):
    variant_examples = []
    for variant, name in (("classic", "metallic mines"), ("flower", "Flower Garden")):
        image, _ = render_game_region(
            lost_game, style="windows7", theme=theme, mine_variant=variant, cell_size=30,
        )
        variant_examples.append((f"Windows 7 {theme} · {name}", image))
    show_row(*variant_examples)

## Window chrome and scene modes

First, wrap one unchanged Classic panel with every chrome option. Aero, modern, and browser chrome provide controlled cross-style comparisons. Next, place the same Microsoft game in grid, game, and desktop modes to compare the framing.

In [ ]:
classic_panel, _ = render_game_region(shared_game, style="classic", cell_size=23)
chrome_examples = []
for chrome in ("none", "classic", "aero", "modern", "browser"):
    chrome_examples.append((f"Classic game · {chrome} chrome", wrap_window(classic_panel, chrome)))
titled_panel = wrap_window(classic_panel, "modern", title="Practice board")
show_row(*chrome_examples[:2], ("Modern chrome · custom title", titled_panel))
show_row(*chrome_examples[2:])

scene_spec = GameSpec(
    seed=42, progress=.48, elapsed_seconds=87,
    appearance={"style": "microsoft", "theme": "blue", "cell_size": 24},
)
grid_scene, _ = render_scene([scene_spec], mode="grid")
game_scene, _ = render_scene([scene_spec], mode="game")
desktop_scene, _ = render_scene([scene_spec], mode="desktop", size=(800, 480), seed=25)
show_row(("Grid mode", grid_scene), ("Game mode", game_scene))
show_row(("Desktop mode · modern chrome · 800 × 480", desktop_scene))

## Desktop compositions and wallpaper sources

The first desktop brings all four default style/chrome pairings together at mixed window sizes. The compact example uses a green Windows 7 Flower Garden game with browser chrome and a generated wallpaper loaded through the image-file option. The negative scene keeps wallpaper, background apps, and the taskbar in view.

In [ ]:
native_windows = [
    GameSpec(seed=42, elapsed_seconds=82, appearance={"style": "classic", "theme": "warm", "cell_size": 25}),
    GameSpec(seed=42, status="lost", progress=.85, elapsed_seconds=148,
             appearance={"style": "windows7", "theme": "blue", "cell_size": 29}),
    GameSpec(seed=61, elapsed_seconds=234, appearance={"style": "microsoft", "theme": "blue", "cell_size": 28}),
    GameSpec(seed=72, elapsed_seconds=31,
             appearance={"style": "google", "theme": "warm", "cell_size": 23, "boundary_ratio": .13}),
]
native_desktop, _ = render_scene(native_windows, size=(1280, 840), seed=7, background_count=3)
show_row(("1280 × 840 · Classic / classic · Windows 7 / Aero · Microsoft / modern · Google / browser", native_desktop))

cross_style = replace(
    native_windows[1], chrome="browser",
    appearance={"style": "windows7", "theme": "green", "mine_variant": "flower",
                "cell_size": 25, "font_family": "Verdana", "font_scale": .8, "bold": False},
)
wallpaper = ImageOps.colorize(Image.linear_gradient("L"), "#102943", "#728657").resize((900, 500))
wallpaper_draw = ImageDraw.Draw(wallpaper)
wallpaper_draw.line(((0, 390), (360, 230), (900, 395)), fill="#9fb387", width=5)
with TemporaryDirectory(prefix="synthetic-showcase-") as directory:
    wallpaper_path = Path(directory) / "generated-wallpaper.png"
    wallpaper.save(wallpaper_path)
    custom_desktop, _ = render_scene(
        [cross_style], size=(720, 485), seed=19, background_count=0, wallpaper_path=wallpaper_path,
    )
show_row(("720 × 485 · green Flower Garden / browser · file-backed wallpaper", custom_desktop))

negative_desktop, _ = render_scene([], size=(920, 500), seed=2, background_count=3)
show_row(("920 × 500 · negative desktop · procedural wallpaper and background apps", negative_desktop))

## Capture augmentation

Every preview starts with one Microsoft game panel. Compare all six Pillow resampling filters at the same scale, then isolate Gaussian blur and JPEG compression. The final capture combines downscaling, blur, and compression. All results are displayed at their actual output sizes.

In [ ]:
capture_spec = GameSpec(
    seed=53, elapsed_seconds=37, mode="targeted", target_number=7,
    appearance={"style": "microsoft", "theme": "warm", "cell_size": 29, "font_family": "Tahoma"},
)
original, capture_meta = render_scene([capture_spec], mode="game")
filters = (
    Image.Resampling.NEAREST, Image.Resampling.BOX, Image.Resampling.BILINEAR,
    Image.Resampling.HAMMING, Image.Resampling.BICUBIC, Image.Resampling.LANCZOS,
)
resampled_examples = []
for resampling in filters:
    resized, _ = augment_scene(original, capture_meta, scale=.60, resampling=resampling)
    resampled_examples.append((f"0.60× · {resampling.name}", resized))
blurred, _ = augment_scene(original, capture_meta, blur=1.15)
jpeg, _ = augment_scene(original, capture_meta, jpeg_quality=35)
combined, _ = augment_scene(
    original, capture_meta, scale=.72, blur=.55, jpeg_quality=65, resampling=Image.Resampling.BICUBIC,
)
show_row(*resampled_examples[:3])
show_row(*resampled_examples[3:])
show_row(("Original", original), ("Blur radius 1.15", blurred), ("JPEG quality 35", jpeg))
show_row(("0.72× bicubic · blur 0.55 · JPEG 65", combined))

## Seeded gallery

These scenes use the existing sampler's defaults. The seeds were chosen to span desktop, game, and grid modes, varied board dimensions, and several statuses. Change the seed list to browse more combinations.

In [ ]:
gallery_seeds = (19, 15, 33, 34, 4, 24)
for seed in gallery_seeds:
    specs, mode = sample_scene(seed)
    image, metadata = render_scene(specs, mode=mode, seed=seed, size=(1152, 760))
    descriptions = []
    for spec, board in zip(specs, metadata["boards"]):
        description = (
            f"{STYLE_NAMES[board['style']]} {board['theme']} · {spec.rows} × {spec.columns} · "
            f"{spec.status} · {board['style_parameters']['cell_size']} px"
        )
        if spec.target_number is not None:
            description += f" · target {spec.target_number}"
        descriptions.append(description)
    summary = " / ".join(descriptions) if descriptions else "No game windows"
    display(Markdown(f"**Seed {seed} · {mode} · {image.width} × {image.height}**  \n{summary}"))
    display(image)